<a href="https://colab.research.google.com/github/taniaSHK-ds/Olist-Sales-Analytics-/blob/main/notebooks/01_cleaning_and_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

files = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geo": "olist_geolocation_dataset.csv",
    "category_map": "product_category_name_translation.csv",
    }

dfs = {
        name : pd.read_csv(path)
        for name , path in files.items()
      }
for name, df in dfs.items():
    print("Table:", name)
    print("Rows and columns:", df.shape)
    print("Empty cells:")
    print(df.isna().sum())
    print("Duplicate rows:", df.duplicated().sum())
    print()


Table: orders
Rows and columns: (99441, 8)
Empty cells:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
Duplicate rows: 0

Table: items
Rows and columns: (112650, 7)
Empty cells:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
Duplicate rows: 0

Table: payments
Rows and columns: (103886, 5)
Empty cells:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
Duplicate rows: 0

Table: reviews
Rows and columns: (99224, 7)
Empty cells:
review_id                      0
order_id                       0
r

In [4]:
dfs["orders"]["order_status"].value_counts()

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [5]:
orders = dfs["orders"].copy()
items = dfs["items"].copy()
payments = dfs["payments"].copy()
reviews = dfs["reviews"].copy()
customers = dfs["customers"].copy()
products = dfs["products"].copy()
sellers = dfs["sellers"].copy()
cat_map = dfs["category_map"].copy()

# keep only delivered orders
orders = orders[orders["order_status"] == "delivered"].copy()
print("Delivered orders:", len(orders))

Delivered orders: 96478


In [6]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

print(orders.isna().sum())

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64


In [7]:
before = len(orders)
orders = orders.dropna(subset=["order_delivered_customer_date"])
print("Removed:", before - len(orders), "| Remaining:", len(orders))

Removed: 8 | Remaining: 96470


In [8]:
products = products.merge(cat_map, on="product_category_name", how="left")

products["category"] = (
    products["product_category_name_english"]
    .fillna(products["product_category_name"])
    .fillna("unknown")
)

products = products[["product_id", "category"]]
print(products["category"].value_counts().head(10))

category
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64


In [9]:
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

reviews = (
    reviews.sort_values("review_answer_timestamp")
    .drop_duplicates("order_id", keep="last")
)

reviews = reviews[["order_id", "review_score"]]
print(len(reviews), reviews["order_id"].is_unique)

98673 True


In [10]:
customers = customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]]
sellers = sellers[["seller_id", "seller_state"]]

df = (
    items.merge(orders, on="order_id", how="inner")
         .merge(customers, on="customer_id", how="left")
         .merge(products, on="product_id", how="left")
         .merge(sellers, on="seller_id", how="left")
         .merge(reviews, on="order_id", how="left")
)
print(df.shape)

(110189, 20)


In [11]:
expected_rows = items["order_id"].isin(orders["order_id"]).sum()
print("Rows in df:", len(df), "| Expected:", expected_rows)
print("Unique orders:", df["order_id"].nunique())
print(df.isna().sum()[df.isna().sum() > 0])

Rows in df: 110189 | Expected: 110189
Unique orders: 96470
order_approved_at                15
order_delivered_carrier_date      1
review_score                    827
dtype: int64


In [12]:
item_total = df.groupby("order_id")[["price", "freight_value"]].sum().sum(axis=1)
paid_total = payments.groupby("order_id")["payment_value"].sum()

compare = pd.concat([item_total.rename("items"), paid_total.rename("paid")], axis=1, join="inner")
print("Item totals:", round(compare["items"].sum()))
print("Payments:", round(compare["paid"].sum()))

Item totals: 15418251
Payments: 15421083


In [13]:
df["order_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)

df["delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
df["estimated_days"] = (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.days

late_gap = df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
df["delay_days"] = (late_gap.dt.total_seconds() / 86400).round(1)
df["is_late"] = (df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]).astype(int)

print(df[["order_month", "delivery_days", "estimated_days", "delay_days", "is_late"]].head())
print("Late orders (share of items):", round(df["is_late"].mean() * 100, 1), "%")

  order_month  delivery_days  estimated_days  delay_days  is_late
0     2017-09              7              15        -8.0        0
1     2017-04             16              18        -2.3        0
2     2018-01              7              21       -13.4        0
3     2018-08              6              11        -5.4        0
4     2017-02             25              40       -15.3        0
Late orders (share of items): 7.9 %


In [14]:
order_df = (
    df.groupby("order_id")
      .agg(
          customer_unique_id=("customer_unique_id", "first"),
          customer_state=("customer_state", "first"),
          order_month=("order_month", "first"),
          purchase_date=("order_purchase_timestamp", "first"),
          items=("order_item_id", "count"),
          items_value=("price", "sum"),
          freight_value=("freight_value", "sum"),
          main_category=("category", "first"),
          delivery_days=("delivery_days", "first"),
          delay_days=("delay_days", "first"),
          is_late=("is_late", "first"),
          review_score=("review_score", "first"),
      )
      .reset_index()
)

order_df["order_value"] = order_df["items_value"] + order_df["freight_value"]
order_df["purchase_date"] = order_df["purchase_date"].dt.date

print(order_df.shape)
print(order_df["order_id"].is_unique)

(96470, 14)
True


In [15]:
print(order_df["order_month"].value_counts().sort_index())

order_month
2016-09       1
2016-10     265
2016-12       1
2017-01     750
2017-02    1653
2017-03    2546
2017-04    2303
2017-05    3545
2017-06    3135
2017-07    3872
2017-08    4193
2017-09    4150
2017-10    4478
2017-11    7288
2017-12    5513
2018-01    7069
2018-02    6555
2018-03    7003
2018-04    6798
2018-05    6749
2018-06    6096
2018-07    6156
2018-08    6351
Name: count, dtype: int64


In [16]:
order_df.to_csv("olist_orders_clean.csv", index=False)

items_small = df[["order_id", "order_month", "category", "customer_state", "seller_state", "price", "freight_value"]]
items_small.to_csv("olist_items_clean.csv", index=False)